# Census data analysis (Snowflake, read-only)

Patient demographics, eligibility, and family links. **Read-only**: `SELECT` / `DESCRIBE` only.

**Cell format:** analysis cells are **SQL cells**, so each result shows Table / Chart / Pivot and the **download** button.

**Config stays in one place.** The Python cell builds table and column names; SQL cells read them as `{{T}}`, `{{C_PT}}`, `{{C_DOB}}`, etc.

**Grain:** the dictionary says one row per patient. Eligibility dates can still produce extra rows, so volume reports both `ROW_COUNT` and `UNIQUE_PATIENTS`.

## 1. Active session

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session

## 2. Config (change names ONLY here)

SQL cells can only read **plain string** variables (`T`, `C_PT`, ...). `{{COL['patient_id']}}` would not work.

Set `QUOTE_COLUMNS = False` if your table uses unquoted uppercase names.

In [ ]:
DATABASE_NAME = "ATTR"
SCHEMA_NAME = "PUBLIC"
TABLE_NAME = "CENSUS"   # try CENSUS, Census, MEMBER_CENSUS

QUOTE_DATABASE = False
QUOTE_SCHEMA = False
QUOTE_TABLE = False
QUOTE_COLUMNS = True

COL = {
    "patient_id": "Member/PatientId",
    "effective_date": "EffectiveDate",
    "termination_date": "TerminationDate",
    "birth_date": "BirthDate",
    "gender": "Gender",
    "first_name": "FirstName",
    "last_name": "LastName",
    "middle_initial": "MiddleInitial",
    "address": "Address",
    "address1": "Address1",
    "address2": "Address2",
    "city": "City",
    "state": "State",
    "phone": "PhoneNumber",
    "zip": "ZipCode",
    "plan_code": "PlanCode",
    "group_code": "GroupCode",
    "plan_type": "PlanType",
    "family_id": "FamilyId",
}


def sf_ident(name, quoted):
    if quoted:
        return '"' + str(name).replace('"', '""') + '"'
    return str(name)


def col(key):
    return sf_ident(COL[key], QUOTE_COLUMNS)


DB = sf_ident(DATABASE_NAME, QUOTE_DATABASE)
T = ".".join(
    [
        DB,
        sf_ident(SCHEMA_NAME, QUOTE_SCHEMA),
        sf_ident(TABLE_NAME, QUOTE_TABLE),
    ]
)

C_PT = col("patient_id")
C_EFF = col("effective_date")
C_TERM = col("termination_date")
C_DOB = col("birth_date")
C_GENDER = col("gender")
C_FN = col("first_name")
C_LN = col("last_name")
C_PLAN = col("plan_code")
C_FAM = col("family_id")

print(f"T = {T}")
print(f"C_PT = {C_PT}")
print(f"C_EFF = {C_EFF}")
print(f"C_TERM = {C_TERM}")
print(f"C_DOB = {C_DOB}")
print(f"C_GENDER = {C_GENDER}")
print(f"C_FN = {C_FN}")
print(f"C_LN = {C_LN}")
print(f"C_PLAN = {C_PLAN}")
print(f"C_FAM = {C_FAM}")

## 3. Find the table (only if the name or schema is wrong)

In [ ]:
SELECT
    CURRENT_ROLE() AS ROLE,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME,
    ROW_COUNT,
    BYTES
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND (
        UPPER(TABLE_NAME) LIKE '%CENSUS%'
     OR UPPER(TABLE_NAME) LIKE '%MEMBER%'
  )
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## 4. Table shape and first 10 rows

In [ ]:
DESCRIBE TABLE {{T}};

In [ ]:
SELECT *
FROM {{T}}
LIMIT 10;

## 5. How many patients?

If `ROW_COUNT` equals `UNIQUE_PATIENTS`, `Member/PatientId` is unique. If row count is higher, the same person has more than one eligibility row.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT({{C_PT}}) AS NON_NULL_PATIENT_ID_ROWS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(*) - COUNT(DISTINCT {{C_PT}}) AS EXTRA_ROWS_VS_UNIQUE_PATIENTS
FROM {{T}};

## 6. Age today, then counts by range

Age is years from `BirthDate` to **today** (`CURRENT_DATE()`), minus 1 if this year's birthday has not happened yet. That is today minus birth date, so ages are positive.

Ranges are non-overlapping: `<1`, `1-10`, `11-20`, … `81+`. Age 10 sits in `1-10`.

If a patient has more than one Census row, this uses one row per patient (latest `EffectiveDate`, then latest `TerminationDate`).

In [ ]:
WITH one_row_per_patient AS (
    SELECT *
    FROM {{T}}
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY {{C_PT}}
        ORDER BY {{C_EFF}} DESC NULLS LAST,
                 {{C_TERM}} DESC NULLS LAST
    ) = 1
),
aged AS (
    SELECT
        CASE
            WHEN {{C_DOB}} IS NULL THEN NULL
            ELSE DATEDIFF('year', {{C_DOB}}, CURRENT_DATE())
                 - IFF(
                       DATEADD(
                           'year',
                           DATEDIFF('year', {{C_DOB}}, CURRENT_DATE()),
                           {{C_DOB}}
                       ) > CURRENT_DATE(),
                       1,
                       0
                   )
        END AS AGE_YEARS
    FROM one_row_per_patient
),
bucketed AS (
    SELECT
        CASE
            WHEN AGE_YEARS IS NULL THEN 90
            WHEN AGE_YEARS < 0 THEN 80
            WHEN AGE_YEARS < 1 THEN 0
            WHEN AGE_YEARS <= 10 THEN 1
            WHEN AGE_YEARS <= 20 THEN 2
            WHEN AGE_YEARS <= 30 THEN 3
            WHEN AGE_YEARS <= 40 THEN 4
            WHEN AGE_YEARS <= 50 THEN 5
            WHEN AGE_YEARS <= 60 THEN 6
            WHEN AGE_YEARS <= 70 THEN 7
            WHEN AGE_YEARS <= 80 THEN 8
            ELSE 9
        END AS SORT_ORDER
    FROM aged
)
SELECT
    SORT_ORDER,
    CASE SORT_ORDER
        WHEN 0 THEN '0 (<1 year)'
        WHEN 1 THEN '1-10'
        WHEN 2 THEN '11-20'
        WHEN 3 THEN '21-30'
        WHEN 4 THEN '31-40'
        WHEN 5 THEN '41-50'
        WHEN 6 THEN '51-60'
        WHEN 7 THEN '61-70'
        WHEN 8 THEN '71-80'
        WHEN 9 THEN '81+'
        WHEN 80 THEN 'Invalid (negative age)'
        ELSE 'Unknown (missing birth date)'
    END AS AGE_RANGE,
    COUNT(*) AS PATIENT_COUNT,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_PATIENTS
FROM bucketed
GROUP BY 1, 2
ORDER BY SORT_ORDER;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    {{C_DOB}} AS BIRTH_DATE,
    CURRENT_DATE() AS TODAY,
    DATEDIFF('year', {{C_DOB}}, CURRENT_DATE())
        - IFF(
              DATEADD(
                  'year',
                  DATEDIFF('year', {{C_DOB}}, CURRENT_DATE()),
                  {{C_DOB}}
              ) > CURRENT_DATE(),
              1,
              0
          ) AS AGE_YEARS
FROM {{T}}
WHERE {{C_DOB}} IS NOT NULL
LIMIT 10;

## 7. Gender — unique values and counts

In [ ]:
SELECT
    {{C_GENDER}} AS GENDER,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM {{T}}
GROUP BY 1
ORDER BY UNIQUE_PATIENTS DESC;

## 8. Family IDs — unique count, size, one family

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT({{C_FAM}}) AS NON_NULL_FAMILY_ID_ROWS,
    COUNT(DISTINCT {{C_FAM}}) AS UNIQUE_FAMILY_IDS,
    COUNT(DISTINCT IFF({{C_FAM}} IS NULL, {{C_PT}}, NULL)) AS PATIENTS_WITH_NULL_FAMILY_ID
FROM {{T}};

In [ ]:
SELECT
    {{C_FAM}} AS FAMILY_ID,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(*) AS ROW_COUNT
FROM {{T}}
WHERE {{C_FAM}} IS NOT NULL
GROUP BY 1
ORDER BY UNIQUE_PATIENTS DESC, ROW_COUNT DESC
LIMIT 25;

In [ ]:
WITH fam AS (
    SELECT
        {{C_FAM}} AS FAMILY_ID,
        COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS
    FROM {{T}}
    WHERE {{C_FAM}} IS NOT NULL
    GROUP BY 1
)
SELECT
    UNIQUE_PATIENTS AS MEMBERS_IN_FAMILY,
    COUNT(*) AS NUMBER_OF_FAMILIES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_FAMILIES
FROM fam
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    {{C_FAM}} AS FAMILY_ID,
    {{C_PT}} AS PATIENT_ID,
    {{C_FN}} AS FIRST_NAME,
    {{C_LN}} AS LAST_NAME,
    {{C_DOB}} AS BIRTH_DATE,
    {{C_GENDER}} AS GENDER,
    {{C_PLAN}} AS PLAN_CODE,
    {{C_EFF}} AS EFFECTIVE_DATE,
    {{C_TERM}} AS TERMINATION_DATE
FROM {{T}}
WHERE {{C_FAM}} = (
        SELECT {{C_FAM}}
        FROM {{T}}
        WHERE {{C_FAM}} IS NOT NULL
        GROUP BY 1
        ORDER BY COUNT(DISTINCT {{C_PT}}) DESC
        LIMIT 1
      )
ORDER BY {{C_DOB}} NULLS LAST, {{C_PT}};

## Notes

- **Downloading:** run a SQL cell, then use the download arrow on that result grid.
- If a cell fails with **invalid identifier**, copy names from `describe_table` into `COL`. `Member/PatientId` needs `QUOTE_COLUMNS = True`.
- Run `config` before the SQL cells.